# OpenAIEmbeddings (2026 업데이트판)

문서 임베딩은 텍스트를 수치 벡터로 변환해 의미를 수치화하는 과정입니다. 이 벡터는 검색, 분류, 군집화, 유사도 계산 등에 활용됩니다.

### 원본 대비 변경 사항
| 항목 | 원본 | 현재 권장 |
|---|---|---|
| LangSmith 설정 | `langchain_teddynote.logging` | 환경 변수 (`LANGSMITH_*`) |
| 모델 생성 | 클래스 직접 생성만 | `init_embeddings("openai:...")` (LangChain v1 통합 초기화) 또는 클래스 직접 생성 |
| 유사도 계산 | `sklearn` 이중 for 문 | `numpy` 행렬 연산 한 번 |

[OpenAI Embeddings 문서](https://platform.openai.com/docs/guides/embeddings)

In [ ]:
%pip install -qU langchain langchain-openai python-dotenv numpy

## 환경 설정

- `.env` 파일의 API 키를 `python-dotenv`로 불러옵니다.
- **변경점**: 책에서 사용한 `langchain_teddynote.logging.langsmith()`는 서드파티 헬퍼입니다. 현재 LangSmith 공식 방식은 환경 변수(`LANGSMITH_TRACING`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT`)만 설정하는 것이며, 별도 패키지가 필요 없습니다.
- 참고: 임베딩 호출(`embed_query`, `embed_documents`)은 Runnable이 아니어서 LangSmith에 트레이스가 남지 않습니다. 이 챕터에서는 없어도 되는 설정이지만, 이후 체인/에이전트 실습과 형태를 맞추기 위해 둡니다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # .env 파일의 키를 환경 변수로 로드

# LangSmith 추적 (LANGSMITH_API_KEY 는 .env 에 넣어 둡니다)
os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "CH08-Embeddings")

## 지원 모델

| MODEL | 기본 차원 | MAX INPUT | 비고 |
|---|---|---|---|
| text-embedding-3-small | 1536 | 8191 | 가성비 기본값 |
| text-embedding-3-large | 3072 | 8191 | 최고 성능 |
| text-embedding-ada-002 | 1536 | 8191 | 레거시 — 신규 프로젝트에서는 사용하지 않음 |

가격·성능 수치는 자주 바뀌므로 공식 문서에서 확인하세요.

## 모델 생성 방법 1: `init_embeddings` (LangChain v1)

LangChain v1부터 `langchain.embeddings` 네임스페이스에 `init_embeddings`가 들어 있습니다. `"공급자:모델명"` 문자열 하나로 모델을 만들 수 있어, 설정 파일에서 공급자를 바꾸기 쉽습니다(채팅 모델의 `init_chat_model`과 같은 방식).

In [ ]:
from langchain.embeddings import init_embeddings

embeddings = init_embeddings("openai:text-embedding-3-small")

## 모델 생성 방법 2: 클래스 직접 생성

공급자 고유 옵션(`dimensions` 등)을 명시적으로 쓰고 싶을 때는 통합 패키지(`langchain-openai`)의 클래스를 직접 생성합니다. 두 방법 모두 결과는 같은 `Embeddings` 인터페이스입니다.

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
text = "임베딩 테스트를 하기 위한 샘플 문장입니다."

## 쿼리 임베딩

`embed_query(text)`는 문자열 하나를 벡터(`list[float]`) 하나로 변환합니다.

In [ ]:
query_result = embeddings.embed_query(text)
query_result[:5]

## Document 임베딩

`embed_documents(texts)`는 문자열 리스트를 받아 벡터 리스트(`list[list[float]]`)를 반환합니다. 여러 문장을 한 번의 API 요청(배치)으로 처리합니다.

In [ ]:
doc_result = embeddings.embed_documents([text, text, text, text])

print(len(doc_result))      # 문서 개수
print(doc_result[0][:5])    # 첫 번째 문서 벡터의 앞 5개 값
print(len(doc_result[0]))   # 차원 (text-embedding-3-small 기본값 1536)

## 차원(dimensions) 조정

`text-embedding-3` 계열은 `dimensions` 파라미터로 출력 차원을 줄일 수 있습니다(Matryoshka 방식). 벡터 저장 용량과 검색 비용이 줄고, 성능 손실은 작습니다.

In [ ]:
embeddings_1024 = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1024)

len(embeddings_1024.embed_query(text))

## 유사도 계산

**변경점**: 원본은 `sklearn.cosine_similarity`를 문장 쌍마다 호출했습니다. 벡터를 한 번에 정규화하고 행렬곱 한 번으로 전체 유사도 행렬을 구하면 코드가 짧고 빠르며, `scikit-learn` 의존성도 필요 없습니다.

In [ ]:
import numpy as np

sentences = [
    "안녕하세요? 반갑습니다.",
    "안녕하세요? 반갑습니다!",
    "안녕하세요? 만나서 반가워요.",
    "Hi, nice to meet you.",
    "I like to eat apples.",
]

vecs = np.array(embeddings_1024.embed_documents(sentences))
vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)  # L2 정규화
sim_matrix = vecs @ vecs.T                                  # 코사인 유사도 행렬

for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        print(f"[유사도 {sim_matrix[i, j]:.4f}] {sentences[i]} \t <=====> \t {sentences[j]}")